In [37]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

FEATURE_NAMES = [
    "diagonal", "height_left", "height_right",
    "margin_low", "margin_up", "length"
]

TARGET_NAME = "is_genuine"

DATA_PATH = os.path.join(os.getcwd(), "data", "billets.csv")

print("Constantes définies !")

Constantes définies !


In [38]:
#chargement
DATA_PATH = r"C:\Users\HP\Desktop\fake-bill-detector\data\billets.csv"  # cellule 1
df = pd.read_csv(DATA_PATH, sep=";")     

In [39]:
#exploration des donnees 
df.describe()

,diagonal,height_left,height_right,margin_low,margin_up,length
count,1500.000000,1500.000000,1500.000000,1463.000000,1500.000000,1500.00000
mean,171.958440,104.029533,103.920307,4.485967,3.151473,112.67850
std,0.305195,0.299462,0.325627,0.663813,0.231813,0.87273
min,171.040000,103.140000,102.820000,2.980000,2.270000,109.49000
25%,171.750000,103.820000,103.710000,4.015000,2.990000,112.03000
50%,171.960000,104.040000,103.920000,4.310000,3.140000,112.96000
75%,172.170000,104.230000,104.150000,4.870000,3.310000,113.34000
max,173.010000,104.880000,104.950000,6.900000,3.910000,114.44000


In [40]:
distribution = df[TARGET_NAME].value_counts()

print("Distribution de la colonne cible (is_genuine) :")
print("-" * 40)
print(f"  Authentiques (True)  : {distribution[True]}")
print(f"  Faux billets (False) : {distribution[False]}")
print("-" * 40)
print(f"  Total                : {len(df)}")
print()

# Pourcentage de chaque classe
pct_vrai = distribution[True]  / len(df) * 100
pct_faux = distribution[False] / len(df) * 100
print(f"  Authentiques : {pct_vrai:.1f}%")
print(f"  Faux         : {pct_faux:.1f}%")

Distribution de la colonne cible (is_genuine) :
----------------------------------------
  Authentiques (True)  : 1000
  Faux billets (False) : 500
----------------------------------------
  Total                : 1500

  Authentiques : 66.7%
  Faux         : 33.3%


In [41]:
valeurs_manquantes = df.isnull().sum()

print("Nombre de valeurs manquantes par colonne :")
print("-" * 40)
print(valeurs_manquantes)
print("-" * 40)
print(f"Total manquant : {valeurs_manquantes.sum()}")

Nombre de valeurs manquantes par colonne :
----------------------------------------
is_genuine       0
diagonal         0
height_left      0
height_right     0
margin_low      37
margin_up        0
length           0
dtype: int64
----------------------------------------
Total manquant : 37


In [42]:
nb_doublons = df.duplicated().sum()

print(f"Nombre de lignes dupliquées : {nb_doublons}")

if nb_doublons == 0:
    print("Aucun doublon détecté — dataset propre.")
else:
    print(f"Attention : {nb_doublons} lignes sont dupliquées.")

Nombre de lignes dupliquées : 0
Aucun doublon détecté — dataset propre.


In [43]:
#nettoyage
lignes_manquantes = df[df["margin_low"].isnull()]

print(f"Lignes avec margin_low manquant : {len(lignes_manquantes)}")
print()
print("Répartition par classe :")
print(lignes_manquantes[TARGET_NAME].value_counts())
print()
print("Moyenne de margin_low par classe (sur les lignes complètes) :")
print(df.groupby(TARGET_NAME)["margin_low"].mean().round(4))


Lignes avec margin_low manquant : 37

Répartition par classe :
is_genuine
True     29
False     8
Name: count, dtype: int64

Moyenne de margin_low par classe (sur les lignes complètes) :
is_genuine
False    5.2159
True     4.1161
Name: margin_low, dtype: float64


In [44]:
# On travaille sur une copie du DataFrame pour ne pas modifier l'original
df_propre = df.copy()

# Calcul de la moyenne de margin_low pour chaque classe
# groupby() regroupe les lignes par classe (True / False)
# transform('mean') recalcule la moyenne et la remet à la bonne ligne
moyenne_par_classe = df_propre.groupby(TARGET_NAME)["margin_low"].transform("mean")

# fillna() remplace les NaN par la valeur calculée
df_propre["margin_low"] = df_propre["margin_low"].fillna(moyenne_par_classe)

print("Imputation terminée.")
print()
print("Vérification — valeurs manquantes après nettoyage :")
print(df_propre.isnull().sum())

Imputation terminée.

Vérification — valeurs manquantes après nettoyage :
is_genuine      0
diagonal        0
height_left     0
height_right    0
margin_low      0
margin_up       0
length          0
dtype: int64


In [45]:
print("margin_low AVANT nettoyage :")
print(df["margin_low"].describe().round(4))
print()
print("margin_low APRES nettoyage :")
print(df_propre["margin_low"].describe().round(4))

margin_low AVANT nettoyage :
count    1463.0000
mean        4.4860
std         0.6638
min         2.9800
25%         4.0150
50%         4.3100
75%         4.8700
max         6.9000
Name: margin_low, dtype: float64

margin_low APRES nettoyage :
count    1500.0000
mean        4.4827
std         0.6597
min         2.9800
25%         4.0300
50%         4.3100
75%         4.8700
max         6.9000
Name: margin_low, dtype: float64


In [46]:
#encodage de la cible
print("AVANT encodage :")
print(f"  Type    : {df_propre[TARGET_NAME].dtype}")
print(f"  Valeurs : {df_propre[TARGET_NAME].unique()}")
print()
print("Aperçu des 5 premières lignes :")
print(df_propre[TARGET_NAME].head())

AVANT encodage :
  Type    : bool
  Valeurs : [ True False]

Aperçu des 5 premières lignes :
0    True
1    True
2    True
3    True
4    True
Name: is_genuine, dtype: bool


In [47]:
df_propre[TARGET_NAME] = df_propre[TARGET_NAME].astype(int)

print("Encodage terminé.")

Encodage terminé.


In [48]:
print("APRES encodage :")
print(f"  Type    : {df_propre[TARGET_NAME].dtype}")
print(f"  Valeurs : {df_propre[TARGET_NAME].unique()}")
print()
print("Répartition :")
print(f"  1 (Authentique) : {(df_propre[TARGET_NAME] == 1).sum()}")
print(f"  0 (Faux billet) : {(df_propre[TARGET_NAME] == 0).sum()}")
print()
print("Aperçu des 5 premières lignes :")
print(df_propre[TARGET_NAME].head())

APRES encodage :
  Type    : int32
  Valeurs : [1 0]

Répartition :
  1 (Authentique) : 1000
  0 (Faux billet) : 500

Aperçu des 5 premières lignes :
0    1
1    1
2    1
3    1
4    1
Name: is_genuine, dtype: int32


In [49]:
#Normalisation + Split train/test
#separation x et y
X = df_propre[FEATURE_NAMES]   # features  → shape (1500, 6)
y = df_propre[TARGET_NAME]     # cible     → shape (1500,)

print("Séparation X / y :")
print(f"  X (features) : {X.shape}")
print(f"  y (cible)    : {y.shape}")
print()
print("Aperçu de X :")
print(X.head())

Séparation X / y :
  X (features) : (1500, 6)
  y (cible)    : (1500,)

Aperçu de X :
   diagonal  height_left  height_right  margin_low  margin_up  length
0    171.81       104.86        104.95        4.52       2.89  112.83
1    171.46       103.36        103.66        3.77       2.99  113.09
2    172.69       104.48        103.50        4.40       2.94  113.16
3    171.36       103.91        103.94        3.62       3.01  113.51
4    171.73       104.28        103.46        4.04       3.48  112.54


In [50]:
# Split train / test 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y        # conserve les proportions 66%/33%
)

print("Split train / test :")
print("-" * 40)
print(f"  X_train : {X_train.shape}  ({len(X_train)} billets)")
print(f"  X_test  : {X_test.shape}   ({len(X_test)} billets)")
print("-" * 40)
print()
print("Proportions dans y_train :")
print(y_train.value_counts(normalize=True).round(3))
print()
print("Proportions dans y_test :")
print(y_test.value_counts(normalize=True).round(3)) 

Split train / test :
----------------------------------------
  X_train : (1200, 6)  (1200 billets)
  X_test  : (300, 6)   (300 billets)
----------------------------------------

Proportions dans y_train :
is_genuine
1    0.667
0    0.333
Name: proportion, dtype: float64

Proportions dans y_test :
is_genuine
1    0.667
0    0.333
Name: proportion, dtype: float64


In [51]:
# Normalisation 
scaler = StandardScaler()

# fit_transform sur X_train : apprend ET normalise
X_train_scaled = scaler.fit_transform(X_train)

# transform sur X_test : applique les mêmes paramètres SANS réapprendre
X_test_scaled  = scaler.transform(X_test)

print("Normalisation terminée.")
print()
print("Vérification sur X_train_scaled :")
print(f"  Moyenne    (doit être ≈ 0) : {X_train_scaled.mean(axis=0).round(4)}")
print(f"  Écart-type (doit être ≈ 1) : {X_train_scaled.std(axis=0).round(4)}") 

Normalisation terminée.

Vérification sur X_train_scaled :
  Moyenne    (doit être ≈ 0) : [ 0. -0.  0. -0.  0. -0.]
  Écart-type (doit être ≈ 1) : [1. 1. 1. 1. 1. 1.]


In [53]:
# 5.4 Récapitulatif final 
print("=" * 50)
print("  PREPROCESSING TERMINÉ — RÉCAPITULATIF")
print("=" * 50)
print(f"  Dataset initial        : {df.shape}")
print(f"  Après nettoyage        : {df_propre.shape}")
print(f"  Valeurs manquantes     : 0 ✅")
print()
print(f"  X_train_scaled : {X_train_scaled.shape}")
print(f"  X_test_scaled  : {X_test_scaled.shape}")
print(f"  y_train        : {y_train.shape}")
print(f"  y_test         : {y_test.shape}")
print()
print("  Prêt pour l'entraînement des modèles ✅")
print("=" * 50) 

  PREPROCESSING TERMINÉ — RÉCAPITULATIF
  Dataset initial        : (1500, 7)
  Après nettoyage        : (1500, 7)
  Valeurs manquantes     : 0 ✅

  X_train_scaled : (1200, 6)
  X_test_scaled  : (300, 6)
  y_train        : (1200,)
  y_test         : (300,)

  Prêt pour l'entraînement des modèles ✅
